*Note: Code has been heavily borrowed from Qiao Jin's GitHub, creator of MedCPT. All relevant credit goes to Qiao Jin.* https://github.com/ncbi/MedCPT

---

# MedCPT

---

In [ ]:
# Load in libraries
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

In [ ]:
CONDITION = "aig"
DATA_DIR = Path(f"../pmc/{CONDITION}/processed")
BATCH_SIZE = 8
QUERY = "diagnostic criteria for autoimmune gastritis"


MedCPT expects a title and absract, so we need to merge titles and with the text 

In [13]:
# read in the articles
articles = pd.read_parquet(DATA_DIR/"articles.parquet")

# read in the passages
passages = pd.read_parquet(DATA_DIR/"passages.parquet")

In [12]:
print("Articles:", articles.columns.tolist())
print("Passages:", passages.columns.tolist())

Articles: ['pmcid', 'title', 'year', 'doi', 'pmid', 'license']
Passages: ['pmcid', 'passage_id', 'year', 'section', 'type', 'text', 'word_count']


In [ ]:
# add each paper's title to its passages
passages = passages.merge(articles[["pmcid", "title"]],
                        on="pmcid",
                        how = "left")

In [ ]:
# Let's clean our data

# Keep passages containing at least one non-whitespace character
passages = passages[
    passages["text"].str.contains(r"\S", na=False)].copy()

# Replace missing article titles with empty strings
passages["title"] =  passages["title"].fillna("")

In [ ]:
# each document contains the title and a passage
# this follows the title-abstract format expected by MedCPT but
# with passages
docs = passages[["title", "text"]].values.tolist()

# Print the first five examples
for doc in docs[:5]:
    print(doc)
    print()

['Autoimmune gastritis with a gastric hamartomatous inverted polyp and two hyperplastic polyps: a case report', 'We report an unusual case of autoimmune gastritis (AIG) complicated with a submucosal tumor (SMT) and two pedunculated polyps in a 60-year-old man. The patient was admitted for epigastric distention, heartburn, and anorexia. Endoscopy showed an SMT in the fundus, two pedunculated polyps in the body, and markedly atrophic mucosa of the body and fundus. The SMT, measuring 20 mm in diameter, was resected by endoscopic submucosal dissection and histologically diagnosed as a gastric hamartomatous inverted polyp (GHIP), which is characterized by submucosal glandular proliferation, cystic dilatation, and calcification. The gland structures consisted of foveolar cells and pseudopyloric or mucous-neck cell types. The two pedunculated polyps that were resected by endoscopic mucosal resection were histologically diagnosed as hyperplastic polyps, which are characterized by hyperplastic 

## MedCPT Cross Encoder

In [ ]:
# Load the MedCPT Cross Encoder
model_name = "ncbi/MedCPT-Cross-Encoder"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Use a GPU if available, otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

In [ ]:
print(model.config.max_position_embeddings)

In [ ]:
# Use the passage text as the document
passages = passages["text"].tolist()
print(passages[:5])

# Combine the query and each passage into pairs
pairs = [[QUERY, passages] for passages in passages]

print(pairs[:5])

['We report an unusual case of autoimmune gastritis (AIG) complicated with a submucosal tumor (SMT) and two pedunculated polyps in a 60-year-old man. The patient was admitted for epigastric distention, heartburn, and anorexia. Endoscopy showed an SMT in the fundus, two pedunculated polyps in the body, and markedly atrophic mucosa of the body and fundus. The SMT, measuring 20 mm in diameter, was resected by endoscopic submucosal dissection and histologically diagnosed as a gastric hamartomatous inverted polyp (GHIP), which is characterized by submucosal glandular proliferation, cystic dilatation, and calcification. The gland structures consisted of foveolar cells and pseudopyloric or mucous-neck cell types. The two pedunculated polyps that were resected by endoscopic mucosal resection were histologically diagnosed as hyperplastic polyps, which are characterized by hyperplastic foveolar glands with pseudopyloric or mucous-neck glands in the inflamed stroma in the mucosa, which consisted 

In [ ]:
# Store the relevance scores from each batch
all_scores = []

# Score all passages in batches
with torch.no_grad():

    for start in tqdm(range(0, len(pairs), BATCH_SIZE)):

        # Select the next batch of query-passage pairs
        batch = pairs[start:start + BATCH_SIZE]

        # Convert the pairs into tokens
        encoded = tokenizer(
            batch,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        ).to(device)

        # Calculate the relevance scores
        logits = model(**encoded).logits.squeeze(dim=1)

        # Move scores to the CPU and store them
        all_scores.append(logits.cpu().numpy())

# Combine all scores
scores = np.concatenate(all_scores)

## MedCPT Article Encoder

In [ ]:
model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder")
tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")

In [ ]:
# use a GPU if available, otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

In [ ]:
# we'll store the embeddings from each batch
all_embeddings = []

In [ ]:
# Generate embeddings for all passages
with torch.no_grad():

    for start in tqdm(range(0, len(docs), BATCH_SIZE)):

        # Select the next batch of passages
        batch = docs[start:start + BATCH_SIZE]

        # Convert the title-passage pairs into tokens
        encoded = tokenizer(
            batch,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        ).to(device)

        # Use the [CLS] token as the passage representation
        embeds = model(**encoded).last_hidden_state[:, 0, :]

        # Move embeddings back to the CPU and store them
        all_embeddings.append(embeds.cpu().numpy())

# Combine all batches into one array
passage_embeddings = np.vstack(all_embeddings)

print(passage_embeddings.shape)

## MedCPT Query Encoder

In [ ]:
# Load the MedCPT query encoder

query_name = "ncbi/MedCPT-Query-Encoder"

query_tokenizer = AutoTokenizer.from_pretained(query_name)
query_model = AutoModel.from_pretrained(query_name).eval()

query = ["What are the diagnositic criteria for autoimmune gastritis?"]

In [ ]:
# Convert the query into tokens

inputs = query_tokenizer(
    query,
    truncation = True,
    padding = True,
    max_length=64,
    return_tensors="pt"
)